# 6) Notes Cleaner Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Turn messy pasted lecture notes into clean Markdown and save to `clean_notes.md`.

## Simple rules

- Accept a block of text.
- Normalize bullets/numbering.
- Save cleaned markdown file.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "06_notes_cleaner_memory.json"
OUTPUT_MD = "clean_notes.md"

def clean_notes(raw: str) -> str:
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    cleaned = []
    n = 1
    for ln in lines:
        if ln.startswith(("-", "*", "•")):
            ln = ln.lstrip("-*•").strip()
            cleaned.append(f"- {ln}")
        else:
            cleaned.append(f"{n}. {ln}")
            n += 1
    return "# Clean Notes\n\n" + "\n".join(cleaned) + "\n"

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}

    Tools.notify("Notes Cleaner Agent is running. Type 'stop' to end.")
    while True:
        raw = Tools.ask("Paste notes block (or stop):")
        if should_stop(raw):
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Bye!")
            break
        cleaned = clean_notes(raw)
        with open(OUTPUT_MD, "w", encoding="utf-8") as f:
            f.write(cleaned)
        memory["last_output_file"] = OUTPUT_MD
        Tools.notify(f"Saved cleaned notes to {OUTPUT_MD}")
        env.execute(Action("save_memory", {}), memory)

run_agent()


## Easy extensions

- Detect headings with ':' and convert to markdown headings.
- Timestamped output files.
- Add a summary template section.